In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [ ]:
# ==========================================
# AIMO3 Gold Medal Contender - Complete Kaggle Solution
# Kaggle Notebook + Model Sources + Metadata Ready
# ==========================================

# ==========================================
# Step 1: Configuration & Imports
# ==========================================
import os
import re
import sys
import time
import random
import ast
import math
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sympy as sp
from sympy import *
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    pipeline, 
    set_seed
)
from concurrent.futures import ThreadPoolExecutor
import contextlib
from io import StringIO

# ==========================================
# Configuration Class
# ==========================================
@dataclass
class AIMOConfig:
    # Model settings (Kaggle Models ready)
    model_path_primary: str = "Qwen/Qwen2.5-Math-7B-Instruct"
    model_path_secondary: str = "deepseek-ai/DeepSeek-Math-7B-Instruct"
    
    # Generation settings
    max_length: int = 2048
    max_new_tokens: int = 1024
    temperature: float = 0.7
    top_p: float = 0.95
    num_beams: int = 1
    do_sample: bool = True
    
    # Self-consistency settings
    num_prompts: int = 4
    samples_per_prompt: int = 8
    max_code_executions: int = 3
    
    # Answer constraints
    answer_mod: int = 100000
    min_answer: int = 0
    max_answer: int = 99999
    
    # Time & safety
    max_time_minutes: int = 540  # 9 hours
    code_timeout: int = 15
    debug: bool = True
    use_gpu: bool = True

cfg = AIMOConfig()

set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("✅ Configuration loaded")

# ==========================================
# Step 2: Competition Data Loading
# ==========================================
def safe_load_test_data():
    candidates = [
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",
        "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv"
    ]
    
    for path in candidates:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"✅ Loaded test data from: {path}")
            print(f"   Shape: {df.shape}")
            return df
    
    raise FileNotFoundError("test.csv not found in expected locations")

test_df = safe_load_test_data()

# Auto-detect problem column
problem_col = next((col for col in ['problem', 'question', 'prompt'] if col in test_df.columns), None)
assert problem_col is not None, "No problem text column found"
print(f"✅ Problem column: '{problem_col}'")

# ==========================================
# Step 3: Safe Code Executor (TIR Core)
# ==========================================
class SafeExecutor:
    def __init__(self, timeout: int = 15):
        self.timeout = timeout
        self.safe_globals = {
            '__builtins__': __builtins__,
            'math': math,
            'sympy': sp,
            'np': np,
            'pi': math.pi,
            'e': math.e
        }
        # Pre-load sympy symbols
        exec("from sympy import *", self.safe_globals)
    
    def execute_code(self, code: str) -> Tuple[str, bool]:
        """Execute code safely with timeout and error capture"""
        stdout = StringIO()
        stderr = StringIO()
        
        def target():
            with contextlib.redirect_stdout(stdout), contextlib.redirect_stderr(stderr):
                try:
                    exec(code, self.safe_globals.copy())
                except Exception as e:
                    print(f"EXEC_ERROR: {type(e).__name__}: {str(e)}", file=stderr)
        
        try:
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(target)
                future.result(timeout=self.timeout)
            return stdout.getvalue().strip() or stderr.getvalue().strip(), True
        except Exception as e:
            return f"TIMEOUT_OR_ERROR: {str(e)}", False

executor = SafeExecutor(cfg.code_timeout)

# ==========================================
# Step 4: Advanced Prompt Engineering
# ==========================================
PROMPT_TEMPLATES = [
    """<|im_start|>system
You are an IMO medalist solving competition math problems.
Use rigorous mathematical reasoning.
For calculations, write Python code in ```python blocks and execute it.
Always box your final integer answer with \\boxed{{ans}}.
Answer must be between 0 and 99999.
<|im_end|>
<|im_start|>user
{problem}<|im_end|>""",
    
    """Solve this olympiad problem step-by-step.
Use SymPy for symbolic math when helpful.
Verify numerical answers with code execution.
Final answer format: \\boxed{{123}}
Problem: {problem}""",
    
    """You are solving a hard AIME/IMO problem.
Think algebraically first, then verify computationally.
Use ```python ... ``` blocks for all non-trivial calculations.
The answer is an integer 0 ≤ ans < 100000.
{problem}""",
    
    """Competition math expert. Solve precisely.
Strategy: (1) classify problem type, (2) algebraic solution, (3) verify with SymPy/Python.
Final answer must be \\boxed{{exact integer}}.
{problem}"""
]

def generate_prompts(problem: str, style_idx: int = 0) -> List[str]:
    """Generate multiple prompt variants"""
    problem = problem.strip()
    prompts = []
    
    for i in range(min(3, cfg.num_prompts)):
        template = PROMPT_TEMPLATES[(style_idx + i) % len(PROMPT_TEMPLATES)]
        prompts.append(template.format(problem=problem))
    
    return prompts

# ==========================================
# Step 5: Robust Answer Extraction
# ==========================================
def extract_answers(text: str) -> List[int]:
    """Extract all plausible integer answers from text"""
    answers = []
    
    # Primary: \boxed{} extraction
    boxed_matches = re.findall(r'\\boxed\{([^}]+)\}', text, re.IGNORECASE)
    for match in boxed_matches:
        match = match.strip()
        ans = parse_candidate(match)
        if ans is not None:
            answers.append(ans)
    
    # Secondary: standalone integers near end
    if not answers:
        tail = text[-500:]
        numbers = re.findall(r'\b(\d{1,5})\b', tail)
        for num in numbers[-3:]:
            ans = parse_candidate(num)
            if ans is not None:
                answers.append(ans)
    
    return answers

def parse_candidate(candidate: str) -> Optional[int]:
    """Parse string candidate to integer safely"""
    try:
        # Direct integer
        if re.match(r'^\d+$', candidate):
            val = int(candidate) % cfg.answer_mod
            return val if cfg.min_answer <= val <= cfg.max_answer else None
        
        # SymPy expression
        expr = sp.sympify(candidate)
        if expr.is_integer:
            val = int(expr) % cfg.answer_mod
            return val if cfg.min_answer <= val <= cfg.max_answer else None
            
    except:
        pass
    return None

# ==========================================
# Step 6: TIR Inference Engine (Core Logic)
# ==========================================
class AIMOReasoner:
    def __init__(self, model_path: str):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading model: {model_path}")
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
        print("✅ Model loaded successfully")
    
    def generate_response(self, prompt: str, temperature: float = 0.7) -> str:
        """Generate response with proper formatting"""
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, 
                              max_length=cfg.max_length).to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=cfg.max_new_tokens,
                temperature=temperature,
                top_p=cfg.top_p,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[inputs['input_ids'].shape:],[1]
                                       skip_special_tokens=True)
        return response
    
    def solve_with_tir(self, problem: str) -> List[int]:
        """Solve using Tool-Integrated Reasoning"""
        all_answers = []
        
        # Generate base responses with different prompts
        prompts = generate_prompts(problem)
        for prompt in prompts:
            response = self.generate_response(prompt, cfg.temperature)
            
            # Extract code blocks for execution
            code_blocks = re.findall(r'```python\s*(.*?)\s*```', response, re.DOTALL)
            for code in code_blocks[-cfg.max_code_executions:]:
                code_out, success = executor.execute_code(code.strip())
                if success:
                    # Feed back execution result
                    feedback_prompt = f"Code output:\n{code_out}\n\nContinue reasoning:"
                    feedback_response = self.generate_response(feedback_prompt, 0.3)
                    response += feedback_response
            
            # Extract all answers
            answers = extract_answers(response)
            all_answers.extend(answers)
        
        return all_answers

reasoner = AIMOReasoner(cfg.model_path_primary)

# ==========================================
# Step 7: Self-Consistency Ensemble
# ==========================================
def majority_vote(answers: List[int]) -> Tuple[int, float, Dict]:
    """Apply self-consistency with confidence scoring"""
    valid_answers = [a for a in answers if a is not None]
    
    if not valid_answers:
        return 0, 0.0, {}
    
    vote_counts = Counter(valid_answers)
    best_answer, best_count = vote_counts.most_common(1)
    confidence = best_count / len(valid_answers)
    
    return best_answer, confidence, dict(vote_counts)

# ==========================================
# Step 8: Main Inference Pipeline
# ==========================================
def solve_problem(problem_text: str) -> Dict:
    """Complete solution pipeline for one problem"""
    start_time = time.time()
    
    # Core TIR solving
    candidates = reasoner.solve_with_tir(problem_text)
    
    # Apply self-consistency
    answer, confidence, votes = majority_vote(candidates)
    
    elapsed = time.time() - start_time
    
    return {
        'answer': int(answer),
        'confidence': float(confidence),
        'num_candidates': len(candidates),
        'elapsed_seconds': elapsed,
        'vote_distribution': votes
    }

# ==========================================
# Step 9: Batch Processing
# ==========================================
print("🚀 Starting AIMO3 Gold Medal pipeline...")
print(f"Total problems: {len(test_df)}")

results = []
global_start = time.time()

for idx, row in test_df.iterrows():
    problem_id = row['id']
    problem_text = row[problem_col]
    
    print(f"\n[{idx+1}/{len(test_df)}] Solving ID={problem_id}")
    
    try:
        result = solve_problem(problem_text)
        result['id'] = problem_id
        results.append(result)
        
        print(f"  → Answer: {result['answer']}, Confidence: {result['confidence']:.1%}")
        print(f"  → Candidates: {result['num_candidates']}, Time: {result['elapsed_seconds']:.1f}s")
        
    except Exception as e:
        print(f"  → ERROR: {str(e)}")
        results.append({
            'id': problem_id,
            'answer': 0,
            'confidence': 0.0,
            'num_candidates': 0,
            'elapsed_seconds': 0.0
        })
    
    # Time safety check
    elapsed_total = (time.time() - global_start) / 60
    if elapsed_total > cfg.max_time_minutes:
        print("⏰ Time limit reached")
        break

# ==========================================
# Step 10: Create Submission
# ==========================================
results_df = pd.DataFrame(results)

submission_df = test_df[['id']].copy()
submission_df = submission_df.merge(
    results_df[['id', 'answer']], 
    on='id', 
    how='left'
)

submission_df['answer'] = submission_df['answer'].fillna(0).astype(int) % cfg.answer_mod
submission_df['answer'] = submission_df['answer'].clip(cfg.min_answer, cfg.max_answer)

# Save submission
output_path = "/kaggle/working/submission.csv"
submission_df.to_csv(output_path, index=False)

print("\n" + "="*60)
print("🎉 SUBMISSION READY!")
print(f"📁 Saved to: {output_path}")
print("\nSubmission preview:")
print(submission_df.head())
print("\nConfidence statistics:")
print(results_df['confidence'].describe())

print("\n" + "="*60)
print("✅ AIMO3 Gold Medal Pipeline Complete!")
print(f"Total runtime: {(time.time() - global_start)/60:.1f} minutes")